In [6]:
# ── Colab Setup (run this first) ──
!pip install pandas tabulate -q

import os

# Option 2: Use Colab Secrets (recommended)
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("AutoDataEngineer")

Notebook 03 — Data Quality Report (The Deliverable)
AutoDataEngineer: Before/After Analysis + Quality Metrics

WHAT THIS DOES:
- Compares raw vs cleaned data across every dimension
- Generates a stakeholder-ready quality report
- Produces the "receipts" that prove the system works

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

PREREQUISITE: Run Notebooks 01 and 02 first

### Setup

In [7]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

# Load both datasets
df_raw = pd.read_csv(DATA_DIR / "raw_data.csv")
df_cleaned = pd.read_csv(DATA_DIR / "cleaned_data.csv")

# Replace empty strings with NaN for fair comparison
df_raw = df_raw.replace({"": np.nan, " ": np.nan})

print(f"Raw data:     {df_raw.shape}")
print(f"Cleaned data: {df_cleaned.shape}")

Raw data:     (200, 14)
Cleaned data: (182, 14)


### Column-by-column null comparison

In [8]:
print("=" * 70)
print("NULL VALUES — BEFORE vs AFTER")
print("=" * 70)

comparison_rows = []
for col in df_raw.columns:
    before = int(df_raw[col].isnull().sum())
    after = int(df_cleaned[col].isnull().sum()) if col in df_cleaned.columns else "DROPPED"
    if isinstance(after, int):
        fixed = before - after
        pct_fixed = f"{(fixed / before * 100):.0f}%" if before > 0 else "N/A"
    else:
        fixed = "—"
        pct_fixed = "—"
    comparison_rows.append({
        "Column": col,
        "Nulls Before": before,
        "Nulls After": after,
        "Fixed": fixed,
        "% Fixed": pct_fixed,
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

total_before = df_raw.isnull().sum().sum()
total_after = df_cleaned.isnull().sum().sum()
print(f"\nTotal: {total_before} nulls → {total_after} nulls ({total_before - total_after} fixed)")

NULL VALUES — BEFORE vs AFTER
                Column  Nulls Before  Nulls After  Fixed % Fixed
          product_name             8            0      8    100%
                brands            21           20      1      5%
            categories            63            0     63    100%
              quantity            45            0     45    100%
          serving_size            25            0     25    100%
           energy_100g            13            0     13    100%
              fat_100g            14            0     14    100%
           sugars_100g            15            0     15    100%
         proteins_100g            22            0     22    100%
    nutrition_grade_fr            43            0     43    100%
             countries            50            0     50    100%
last_modified_datetime            42          173   -131   -312%
                  code             8            0      8    100%
             packaging            47            0     47    

### Duplicate analysis

In [9]:
print("\n" + "=" * 70)
print("DUPLICATES — BEFORE vs AFTER")
print("=" * 70)

dup_before = df_raw.duplicated().sum()
dup_after = df_cleaned.duplicated().sum()
print(f"  Duplicate rows before: {dup_before}")
print(f"  Duplicate rows after:  {dup_after}")
print(f"  Removed: {dup_before - dup_after}")


DUPLICATES — BEFORE vs AFTER
  Duplicate rows before: 18
  Duplicate rows after:  0
  Removed: 18


### Data type consistency check

In [10]:
print("\n" + "=" * 70)
print("DATA TYPES — BEFORE vs AFTER")
print("=" * 70)

for col in df_cleaned.columns:
    before_type = str(df_raw[col].dtype) if col in df_raw.columns else "N/A"
    after_type = str(df_cleaned[col].dtype)
    changed = " ← FIXED" if before_type != after_type else ""
    print(f"  {col:30s} {before_type:15s} → {after_type:15s}{changed}")


DATA TYPES — BEFORE vs AFTER
  product_name                   object          → object         
  brands                         object          → object         
  categories                     object          → object         
  quantity                       object          → object         
  serving_size                   object          → object         
  energy_100g                    object          → float64         ← FIXED
  fat_100g                       object          → float64         ← FIXED
  sugars_100g                    object          → float64         ← FIXED
  proteins_100g                  object          → float64         ← FIXED
  nutrition_grade_fr             object          → object         
  countries                      object          → object         
  last_modified_datetime         object          → object         
  code                           float64         → float64        
  packaging                      object          → object         


### Value consistency check

In [11]:
print("\n" + "=" * 70)
print("VALUE CONSISTENCY — SAMPLE CHECKS")
print("=" * 70)

# Check a few columns for standardization
for col in ["nutrition_grade_fr", "countries", "brands", "categories", "packaging"]:
    if col in df_cleaned.columns:
        unique_before = df_raw[col].dropna().nunique() if col in df_raw.columns else 0
        unique_after = df_cleaned[col].dropna().nunique()
        print(f"\n  {col}:")
        print(f"    Unique values: {unique_before} → {unique_after}")
        if unique_after <= 15:
            print(f"    Clean values:  {sorted(df_cleaned[col].dropna().unique().tolist())}")


VALUE CONSISTENCY — SAMPLE CHECKS

  nutrition_grade_fr:
    Unique values: 8 → 6
    Clean values:  ['a', 'b', 'c', 'd', 'e', 'unknown']

  countries:
    Unique values: 7 → 7
    Clean values:  ['FR', 'FRANCE', 'GERMANY', 'UNITED STATES', 'UNKNOWN', 'US', 'USA']

  brands:
    Unique values: 9 → 7
    Clean values:  ['freshfarm', 'greenvalley', 'healthychoice', 'natureco', 'organic life', 'organiclife', 'pureeats']

  categories:
    Unique values: 6 → 5
    Clean values:  ['beverages', 'dairy', 'dairy products', 'snacks', 'unknown']

  packaging:
    Unique values: 6 → 5
    Clean values:  ['cardboard', 'cardboard box', 'glass', 'plastic', 'unknown']


### Numeric column validation

In [12]:
print("\n" + "=" * 70)
print("NUMERIC COLUMNS — VALIDATION")
print("=" * 70)

for col in ["energy_100g", "fat_100g", "sugars_100g", "proteins_100g"]:
    if col in df_cleaned.columns:
        print(f"\n  {col}:")
        # Try to convert to numeric
        numeric_vals = pd.to_numeric(df_cleaned[col], errors="coerce")
        non_numeric = numeric_vals.isnull().sum() - df_cleaned[col].isnull().sum()
        print(f"    Non-numeric values remaining: {max(0, non_numeric)}")
        valid = numeric_vals.dropna()
        if len(valid) > 0:
            print(f"    Range: {valid.min():.1f} — {valid.max():.1f}")
            print(f"    Mean:  {valid.mean():.1f}")
            negatives = (valid < 0).sum()
            if negatives > 0:
                print(f"    WARNING: {negatives} negative values")


NUMERIC COLUMNS — VALIDATION

  energy_100g:
    Non-numeric values remaining: 0
    Range: 0.0 — 890.6
    Mean:  443.9

  fat_100g:
    Non-numeric values remaining: 0
    Range: 0.0 — 49.9
    Mean:  22.3

  sugars_100g:
    Non-numeric values remaining: 0
    Range: 0.0 — 60.0
    Mean:  27.0

  proteins_100g:
    Non-numeric values remaining: 0
    Range: 0.0 — 29.9
    Mean:  13.5


### Generate the full quality report

In [13]:
print("\n" + "=" * 70)
print("FINAL DATA QUALITY REPORT")
print("=" * 70)

report = {
    "dataset": {
        "rows_before": len(df_raw),
        "rows_after": len(df_cleaned),
        "columns_before": len(df_raw.columns),
        "columns_after": len(df_cleaned.columns),
    },
    "nulls": {
        "total_before": int(total_before),
        "total_after": int(total_after),
        "fixed": int(total_before - total_after),
        "fix_rate": f"{(total_before - total_after) / max(total_before, 1) * 100:.1f}%",
    },
    "duplicates": {
        "before": int(dup_before),
        "after": int(dup_after),
        "removed": int(dup_before - dup_after),
    },
    "quality_score": {},
}

# Calculate per-column quality score
scores = []
for col in df_cleaned.columns:
    col_score = 100
    null_pct = df_cleaned[col].isnull().mean() * 100
    col_score -= null_pct  # Deduct for nulls
    scores.append({"column": col, "score": round(max(0, col_score), 1)})

avg_score = sum(s["score"] for s in scores) / len(scores)
report["quality_score"] = {
    "overall": round(avg_score, 1),
    "per_column": scores,
}

print(f"""
  Rows:              {report['dataset']['rows_before']} → {report['dataset']['rows_after']}
  Nulls fixed:       {report['nulls']['fixed']} ({report['nulls']['fix_rate']})
  Duplicates removed:{report['duplicates']['removed']}
  Quality score:     {report['quality_score']['overall']}/100

  Per-column scores:""")

for s in scores:
    bar = "█" * int(s["score"] / 5)
    print(f"    {s['column']:30s} {s['score']:5.1f}/100 {bar}")


FINAL DATA QUALITY REPORT

  Rows:              200 → 182
  Nulls fixed:       223 (53.6%)
  Duplicates removed:18
  Quality score:     92.4/100

  Per-column scores:
    product_name                   100.0/100 ████████████████████
    brands                          89.0/100 █████████████████
    categories                     100.0/100 ████████████████████
    quantity                       100.0/100 ████████████████████
    serving_size                   100.0/100 ████████████████████
    energy_100g                    100.0/100 ████████████████████
    fat_100g                       100.0/100 ████████████████████
    sugars_100g                    100.0/100 ████████████████████
    proteins_100g                  100.0/100 ████████████████████
    nutrition_grade_fr             100.0/100 ████████████████████
    countries                      100.0/100 ████████████████████
    last_modified_datetime           4.9/100 
    code                           100.0/100 ██████████████████

### Generate Markdown report

In [14]:
md = "# AutoDataEngineer — Data Quality Report\n\n"
md += "---\n\n"
md += "## Overview\n\n"
md += f"| Metric | Before | After | Change |\n"
md += f"|--------|--------|-------|--------|\n"
md += f"| Rows | {report['dataset']['rows_before']} | {report['dataset']['rows_after']} | {report['dataset']['rows_before'] - report['dataset']['rows_after']} removed |\n"
md += f"| Null values | {report['nulls']['total_before']} | {report['nulls']['total_after']} | {report['nulls']['fixed']} fixed |\n"
md += f"| Duplicate rows | {report['duplicates']['before']} | {report['duplicates']['after']} | {report['duplicates']['removed']} removed |\n"
md += f"| Quality score | — | {report['quality_score']['overall']}/100 | — |\n\n"

md += "## Column Quality Scores\n\n"
md += "| Column | Score | Status |\n"
md += "|--------|-------|--------|\n"
for s in scores:
    status = "Excellent" if s["score"] >= 90 else ("Good" if s["score"] >= 70 else "Needs Work")
    md += f"| {s['column']} | {s['score']}/100 | {status} |\n"

md += f"\n---\n\n*Generated by AutoDataEngineer — autonomous multi-agent data cleaning system*\n"

# Display
try:
    from IPython.display import display, Markdown
    display(Markdown(md))
except ImportError:
    print(md)

# Save report
with open(DATA_DIR / "quality_report.json", "w") as f:
    json.dump(report, f, indent=2)

with open(DATA_DIR / "quality_report.md", "w") as f:
    f.write(md)

print(f"\nSaved: quality_report.json + quality_report.md")
print("Notebook 03 complete!")

# AutoDataEngineer — Data Quality Report

---

## Overview

| Metric | Before | After | Change |
|--------|--------|-------|--------|
| Rows | 200 | 182 | 18 removed |
| Null values | 416 | 193 | 223 fixed |
| Duplicate rows | 18 | 0 | 18 removed |
| Quality score | — | 92.4/100 | — |

## Column Quality Scores

| Column | Score | Status |
|--------|-------|--------|
| product_name | 100.0/100 | Excellent |
| brands | 89.0/100 | Good |
| categories | 100.0/100 | Excellent |
| quantity | 100.0/100 | Excellent |
| serving_size | 100.0/100 | Excellent |
| energy_100g | 100.0/100 | Excellent |
| fat_100g | 100.0/100 | Excellent |
| sugars_100g | 100.0/100 | Excellent |
| proteins_100g | 100.0/100 | Excellent |
| nutrition_grade_fr | 100.0/100 | Excellent |
| countries | 100.0/100 | Excellent |
| last_modified_datetime | 4.9/100 | Needs Work |
| code | 100.0/100 | Excellent |
| packaging | 100.0/100 | Excellent |

---

*Generated by AutoDataEngineer — autonomous multi-agent data cleaning system*



Saved: quality_report.json + quality_report.md
Notebook 03 complete!
